# Typing: аннотации типов

1. Динамическая типизация
2. Аннотации — подсказка, а не проверка
3. Аннотации переменных и коллекций
4. `Any` и `object`
5. `Optional`: значение может отсутствовать
6. `Union`
7. `Callable`
8. Псевдонимы типов и обобщённые функции (Python 3.12+)

Аннотации проверяет не интерпретатор, а статический анализатор: **mypy** или PyCharm.
mypy не умеет проверять `.ipynb` напрямую, поэтому тот же код лежит в `typing_demo.py`:

```
uv run mypy support_materials/typing_demo.py
```

# Динамическая типизация

Тип принадлежит значению, а не переменной: одну и ту же переменную можно перепривязать к значению другого типа.

In [ ]:
value = 8
print(type(value))

value = "8"
print(type(value))

# Аннотации — подсказка, а не проверка

Синтаксис: `параметр: тип`, возвращаемое значение — `-> тип`.

In [ ]:
def double(number: int) -> int:
    """Удваивает число."""
    return number * 2


double(8)

Интерпретатор аннотации не проверяет: строка «умножится» на 2 без ошибки.

Ошибку найдёт только статический анализатор:
```
error: Argument 1 to "double" has incompatible type "str"; expected "int"  [arg-type]
```

In [ ]:
double("88")

Аннотации хранятся в функции и доступны во время выполнения, но Python их никак не использует:

In [ ]:
double.__annotations__

# Аннотации переменных

Для коллекций указываем тип элементов в квадратных скобках.

In [ ]:
a: bool = True
b: int = 2
c: str = "something"
d: list[int] = [1, 2, 3]  # список целых чисел
e: dict[str, float] = {}  # для пустого словаря аннотация особенно полезна
f: list[dict[str, int]] = [{}, {}]  # вложенные типы
g: tuple[str, float] = ("MW", 180.16)  # кортеж фиксированной длины
h: tuple[int, ...] = (1, 2, 3)  # кортеж любой длины
i: set[str] = {"C", "N", "O"}

# `Any` и `object`

`Any` — «любой тип», проверка для такого значения **отключается**.

In [ ]:
from typing import Any


def print_everything(item: Any) -> None:
    """Печатает что угодно."""
    print(item)


def shout(item: Any) -> str:
    """С Any mypy молчит, даже если у объекта нет такого метода."""
    return item.upper()


print_everything([1, "two", 3.0])
shout("benzene")

mypy не предупредил, но при вызове с `int` код упадёт:

In [ ]:
shout(42)

`object` тоже принимает что угодно, но анализатор потребует проверить тип перед использованием.

Без проверки mypy скажет: `"object" has no attribute "upper"`.

In [ ]:
def shout_safe(item: object) -> str:
    """Принимает любой объект, но mypy требует проверки типа."""
    if isinstance(item, str):
        return item.upper()  # здесь mypy уже знает, что item — str
    return str(item)


shout_safe("benzene"), shout_safe(42)

# `Optional`: значение может отсутствовать

`Chem.MolFromSmiles` возвращает `None` для некорректного SMILES.

`Optional[X]` из `typing` и `X | None` (Python 3.10+) — одно и то же.
Если функция возвращает значение не во всех ветках, mypy выдаст `Missing return statement`.

In [ ]:
from rdkit import Chem


def parse_smiles(smiles: str) -> Chem.Mol | None:
    """Возвращает молекулу или None, если SMILES некорректен."""
    molecule = Chem.MolFromSmiles(smiles)
    if molecule is None:
        print(f"Некорректный SMILES: {smiles}")
        return None
    return molecule


print(parse_smiles("c1ccccc1"))
print(parse_smiles("not_a_smiles"))

**Сужение типа (type narrowing):** после проверки `is None` анализатор знает, что дальше — `Chem.Mol`.

Без проверки mypy скажет: `Item "None" of "Mol | None" has no attribute "GetNumAtoms"`.

In [ ]:
def count_atoms(smiles: str) -> int:
    """Считает тяжёлые атомы."""
    molecule = parse_smiles(smiles)
    if molecule is None:
        return 0
    return molecule.GetNumAtoms()


print(count_atoms("c1ccccc1"))
print(count_atoms("not_a_smiles"))

# `Union`: одно из нескольких значений

`int | str` — то же, что `Union[int, str]` из `typing`.

In [ ]:
def to_float(value: int | str) -> float:
    """Принимает int или str."""
    return float(value)


to_float(3), to_float("2.5")

# `Callable`: функция как аргумент

`Callable[[типы аргументов], тип результата]`

In [ ]:
from collections.abc import Callable


def apply_to_all(func: Callable[[str], int], items: list[str]) -> list[int]:
    """Применяет функцию к каждому элементу списка."""
    return [func(item) for item in items]


apply_to_all(count_atoms, ["CCO", "c1ccccc1"])

# Псевдонимы типов и обобщённые функции (Python 3.12+)

`type` даёт имя длинному типу.

In [ ]:
type Descriptors = dict[str, float]


def mol_weight(descriptors: Descriptors) -> float:
    """Достаёт молекулярную массу из словаря дескрипторов."""
    return descriptors["MW"]


mol_weight({"MW": 46.07, "LogP": -0.14})

Обобщённая функция: `T` — переменная типа. Тип результата совпадает с типом элементов списка.

In [ ]:
def first[T](items: list[T]) -> T:
    """Возвращает первый элемент списка."""
    return items[0]


print(first(["CCO", "CCN"]))  # mypy знает, что это str
print(first([1, 2, 3]))  # а здесь — int